# Data Cleaning Notebook

This notebook loads the raw taxi and weather data from GCS, performs data cleaning using pandas, and saves the cleaned data back to GCS.

In [ ]:
pip install pandas pyarrow fastparquet

In [ ]:
import os
import pandas as pd
import json
from dotenv import load_dotenv
from google.oauth2 import service_account
from google.cloud import storage
import io

# Load environment variables
load_dotenv()
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")

# GCS client setup
credentials = service_account.Credentials.from_service_account_file(service_account_key)
client = storage.Client(project=project_id, credentials=credentials)
bucket = client.bucket(bucket_name)

## Load Weather Data from GCS

In [ ]:
# Load weather data JSON from GCS
weather_blob = bucket.blob("noaa_weather_data.json")
weather_data = json.loads(weather_blob.download_as_text())
weather_df = pd.DataFrame(weather_data)
print(f"Weather data shape: {weather_df.shape}")
weather_df.head()

## Clean Weather Data

In [ ]:
# Convert date to datetime
weather_df['date'] = pd.to_datetime(weather_df['date'])

# Convert values from tenths to actual units (example for temperature)
# Note: Adjust based on datatype - TMAX/TMIN are in tenths of degrees C
weather_df['value'] = weather_df['value'] / 10.0

# Pivot to have datatypes as columns
weather_clean = weather_df.pivot_table(
    index='date', 
    columns='datatype', 
    values='value',
    aggfunc='first'
).reset_index()

# Rename columns for clarity
weather_clean = weather_clean.rename(columns={
    'PRCP': 'precipitation_mm',
    'SNOW': 'snowfall_mm',
    'SNWD': 'snow_depth_mm',
    'TMAX': 'max_temp_c',
    'TMIN': 'min_temp_c'
})

print(f"Cleaned weather data shape: {weather_clean.shape}")
weather_clean.head()

## Load Taxi Data from GCS

Note: This example loads one taxi file. You'll need to load and combine all relevant files.

In [ ]:
# Example: Load one taxi parquet file from GCS
# List all blobs with prefix 'raw-taxi/'
taxi_blobs = list(bucket.list_blobs(prefix='raw-taxi/'))
print(f"Found {len(taxi_blobs)} taxi data files")

# Load first file as example
if taxi_blobs:
    taxi_blob = taxi_blobs[0]
    buffer = io.BytesIO()
    taxi_blob.download_to_file(buffer)
    buffer.seek(0)
    try:
        taxi_df = pd.read_parquet(buffer, engine='fastparquet')
        print(f"Taxi data shape: {taxi_df.shape}")
        print(taxi_df.head())
    except Exception as e:
        print(f"Error reading parquet: {e}")
        print("Parquet reading failed. Consider re-downloading data as CSV.")
        taxi_df = pd.DataFrame()
else:
    print("No taxi data files found")
    taxi_df = pd.DataFrame()

## Clean Taxi Data

In [ ]:
# Example cleaning operations for TLC Yellow Taxi data
if not taxi_df.empty:
    # Check column names first
    print("Columns in taxi data:", taxi_df.columns.tolist())
    
    # Use correct column names for TLC data
    datetime_cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime']
    fare_col = 'fare_amount'
    distance_col = 'trip_distance'
    
    # Remove rows with missing critical data
    taxi_df = taxi_df.dropna(subset=datetime_cols + [fare_col])
    
    # Convert datetime columns
    for col in datetime_cols:
        taxi_df[col] = pd.to_datetime(taxi_df[col])
    
    # Filter out invalid fares
    taxi_df = taxi_df[(taxi_df[fare_col] > 0) & (taxi_df[fare_col] < 500)]
    
    # Filter out invalid trip distances
    taxi_df = taxi_df[(taxi_df[distance_col] > 0) & (taxi_df[distance_col] < 100)]
    
    # Add derived columns
    taxi_df['trip_duration_minutes'] = (taxi_df['tpep_dropoff_datetime'] - taxi_df['tpep_pickup_datetime']).dt.total_seconds() / 60
    
    print(f"Cleaned taxi data shape: {taxi_df.shape}")
    taxi_df.head()
else:
    print("No taxi data to clean")

## Save Cleaned Data Back to GCS

In [ ]:
# Save cleaned weather data
weather_output_blob = bucket.blob("cleaned_weather_data.parquet")
buffer = io.BytesIO()
weather_clean.to_parquet(buffer, index=False)
buffer.seek(0)
weather_output_blob.upload_from_file(buffer, content_type='application/octet-stream')
print("Cleaned weather data saved to GCS")

# Save cleaned taxi data (example for one file)
if not taxi_df.empty:
    taxi_output_blob = bucket.blob("cleaned_taxi_sample.parquet")
    buffer = io.BytesIO()
    taxi_df.to_parquet(buffer, index=False)
    buffer.seek(0)
    taxi_output_blob.upload_from_file(buffer, content_type='application/octet-stream')
    print("Cleaned taxi data saved to GCS")